In [3]:
import os
os.environ["HF_TOKEN"] = os.getenv("HUGGINGFACE_API_KEY")
# Get token from → https://huggingface.co/settings/tokens

#### Resume Chunking 

In [2]:
def chunk_resume(parsed_resume):
    chunks = []
    name = parsed_resume.get("name") or "Candidate"

    # ── Chunk 1 — Summary ─────────────────────────────────────
    # Rich sentence format — sets context for the whole resume
    summary = parsed_resume.get("summary") or ""
    if summary:
        chunks.append({
            "type"    : "summary",
            "content" : (
                f"{name} is a professional with the following background: "
                f"{summary}"
            )
        })

    # ── Chunk 2 — Skills ──────────────────────────────────────
    # Split into core + additional to avoid one long token dump
    # Sentence format embeds much better than comma-separated list
    skills = parsed_resume.get("skills") or []
    if skills:
        core_skills  = skills[:10]
        extra_skills = skills[10:]

        content = (
            f"{name} is proficient in the following core technologies "
            f"and tools: {', '.join(core_skills)}."
        )
        if extra_skills:
            content += (
                f" Additional skills and expertise include: "
                f"{', '.join(extra_skills)}."
            )
        chunks.append({
            "type"    : "skills",
            "content" : content
        })

    # ── Chunk 3 — Experience + Responsibilities (separated) ───
    # Experience = who/where/when  →  for seniority & domain matching
    # Responsibilities = what they did  →  for JD responsibilities matching
    experience = parsed_resume.get("experience") or []
    if experience:
        exp_lines  = []
        resp_lines = []

        for exp in experience:
            company = exp.get("company")    or "Unknown Company"
            title   = exp.get("title")      or "Unknown Title"
            start   = exp.get("start_date") or ""
            end     = exp.get("end_date")   or "Present"
            resps   = exp.get("responsibilities") or []

            # Experience line — role context
            exp_lines.append(
                f"{name} worked as {title} at {company} "
                f"from {start} to {end}."
            )

            # Responsibilities lines — actual work done
            for r in resps:
                resp_lines.append(f"- {r}")

        # One combined experience chunk
        chunks.append({
            "type"    : "experience",
            "content" : " ".join(exp_lines)
        })

        # Separate responsibilities chunk — matches JD responsibilities
        if resp_lines:
            chunks.append({
                "type"    : "responsibilities",
                "content" : (
                    f"{name} has the following professional responsibilities "
                    f"and achievements:\n" + "\n".join(resp_lines)
                )
            })

    # ── Chunk 4 — Projects ────────────────────────────────────
    # One chunk per project for granular matching
    projects = parsed_resume.get("projects") or []
    for project in projects:
        proj_name = project.get("name")        or "Unnamed Project"
        desc      = project.get("description") or ""
        tech      = project.get("tech")        or []

        chunks.append({
            "type"    : "project",
            "content" : (
                f"{name} built a project called {proj_name}. "
                f"{desc} "
                f"Technologies and tools used: {', '.join(tech)}."
            )
        })

    # ── Chunk 5 — Education ───────────────────────────────────
    education = parsed_resume.get("education") or []
    if education:
        edu_lines = []
        for edu in education:
            degree      = edu.get("degree")      or ""
            institution = edu.get("institution") or ""
            year        = edu.get("year")        or ""
            edu_lines.append(
                f"{name} studied {degree} at {institution} ({year})."
            )
        chunks.append({
            "type"    : "education",
            "content" : " ".join(edu_lines)
        })

    # ── Chunk 6 — Certifications → stored as qualifications ───
    # Mapped to "qualifications" to match CHUNK_WEIGHTS key
    # Sentence format lists them as professional qualifications
    certifications = parsed_resume.get("certifications") or []
    if certifications:
        chunks.append({
            "type"    : "qualifications",
            "content" : (
                f"{name} has demonstrated professional qualifications "
                f"through the following certifications and credentials: "
                f"{', '.join(certifications)}."
            )
        })

    # ── Chunk 7 — Contacts ────────────────────────────────────
    # Weight = 0.00 so doesn't affect score
    # Still stored for display in Phase 7 Streamlit UI
    contacts = parsed_resume.get("contact") or {}
    if contacts:
        email    = contacts.get("email")    or ""
        phone    = contacts.get("phone")    or ""
        location = contacts.get("location") or ""
        linkedin = contacts.get("linkedin") or ""

        chunks.append({
            "type"    : "contacts",
            "content" : (
                f"{name} can be contacted at {email} or {phone}. "
                f"Located in {location}. "
                f"LinkedIn profile: {linkedin}."
            )
        })

    return chunks

In [3]:
import json
import os

ALL_RESUMES_PATH = r"..\output_jsons_resume\all_resumes.json"
ALL_RESUME_CHUNKS_PATH = r"..\chunks\all_resume_chunks.json"


def load_json_file(path, default=None):
    if default is None:
        default = {}

    if not os.path.exists(path):
        return default

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json_file(path, data):
    folder = os.path.dirname(path)
    if folder:
        os.makedirs(folder, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)


def chunk_new_resumes(
    all_resumes_path=ALL_RESUMES_PATH,
    all_chunks_path=ALL_RESUME_CHUNKS_PATH
):
    all_resumes = load_json_file(all_resumes_path, {})
    all_resume_chunks = load_json_file(all_chunks_path, {})
    new_resume_chunks = {}

    already_chunked = 0
    newly_chunked = 0
    failed = 0

    for filename, resume_data in all_resumes.items():
        if filename in all_resume_chunks:
            already_chunked += 1
            print(f"Skipping already chunked resume: {filename}")
            continue

        try:
            chunks = chunk_resume(resume_data)

            if not chunks:
                failed += 1
                print(f"No chunks created for: {filename}")
                continue

            all_resume_chunks[filename] = chunks
            new_resume_chunks[filename] = chunks
            newly_chunked += 1

            print(f"Chunked new resume: {filename} -> {len(chunks)} chunks")

        except Exception as e:
            failed += 1
            print(f"Failed to chunk {filename}: {e}")

    save_json_file(all_chunks_path, all_resume_chunks)

    print("\nChunking Summary")
    print(f"Already chunked: {already_chunked}")
    print(f"Newly chunked: {newly_chunked}")
    print(f"Failed: {failed}")
    print(f"Total resumes in all_resume_chunks.json: {len(all_resume_chunks)}")

    return all_resume_chunks, new_resume_chunks


all_resume_chunks, new_resume_chunks = chunk_new_resumes()


Chunked new resume: Abhishek_Shaurya_Resume.pdf -> 11 chunks
Chunked new resume: Jay Kumar Behera_CV_DS.pdf -> 10 chunks
Chunked new resume: Komal_Kamble_1 (1).pdf -> 7 chunks
Chunked new resume: MeetLad_Resume.pdf -> 7 chunks
Chunked new resume: Prity-Kumari-Resume-DevOps-2025.pdf -> 8 chunks
Chunked new resume: SOUMYADEEP_SEN_CV.pdf -> 8 chunks

Chunking Summary
Already chunked: 0
Newly chunked: 6
Failed: 0
Total resumes in all_resume_chunks.json: 6


In [5]:
all_resume_chunks

{'Abhishek_Shaurya_Resume.pdf': [{'type': 'summary',
   'content': 'Abhishek Shaurya is a professional with the following background: Product Owner with 3 years driving platform and financial system modernization across 90+ enterprise retail clients (NA, EMEA, APAC). Owns roadmap, backlog, and delivery across ETL re-architecture, reporting platform migration, and a 0->1 AI audit assistant - saving ~7,500 engineering hours annually and surfacing $4-5M in financial risk exposure. Targeting senior PM roles in B2B SaaS, fintech, or data/AI products.'},
  {'type': 'skills',
   'content': 'Abhishek Shaurya is proficient in the following core technologies and tools: Roadmap Ownership, Feature Prioritization (RICE, MoSCoW), Platform Architecture, PRD/BRD, MVP Scoping, Financial Systems Modeling, User Research, SQL, Python, Snowflake. Additional skills and expertise include: MicroStrategy, ETL Design, AWS Lambda, Figma, Postman, Agile/Scrum, Sprint Planning, Backlog Ownership, Stakeholder Manag

#### JD Chunking

In [4]:
def chunk_jd(parsed_jd):
    chunks = []
    job_title = parsed_jd.get("job_title") or "Role"
    company   = parsed_jd.get("company")   or "the company"
    location  = parsed_jd.get("location")  or ""
    exp       = parsed_jd.get("experience_required") or ""
    emp_type  = parsed_jd.get("employment_type")     or ""

    # ── Chunk 1 — Summary ─────────────────────────────────────
    # Mirrors resume "summary" chunk type → direct comparison
    # Was "overview" before — rename to "summary" to match
    # CHUNK_WEIGHTS and resume chunk types
    summary = parsed_jd.get("job_summary") or ""
    if summary:
        chunks.append({
            "type"    : "summary",
            "content" : (
                f"We are hiring a {job_title} at {company}"
                f"{f' located in {location}' if location else ''}. "
                f"{summary} "
                f"This is a {emp_type} role requiring {exp} of experience."
            )
        })

    # ── Chunk 2 — Skills ──────────────────────────────────────
    # Sentence format — mirrors resume skills chunk
    # Must-have weighted higher in text to signal importance
    skills       = parsed_jd.get("skills_required") or {}
    must_have    = skills.get("must_have")    or []
    good_to_have = skills.get("good_to_have") or []

    if must_have or good_to_have:
        content = ""
        if must_have:
            content += (
                f"The ideal candidate for {job_title} must be proficient "
                f"in the following technologies and tools: "
                f"{', '.join(must_have)}. "
            )
        if good_to_have:
            content += (
                f"Knowledge of the following is a strong advantage: "
                f"{', '.join(good_to_have)}."
            )
        chunks.append({
            "type"    : "skills",
            "content" : content.strip()
        })

    # ── Chunk 3 — Responsibilities ────────────────────────────
    # Richer sentence format per category
    # Mirrors resume "responsibilities" chunk → strong cosine match
    responsibilities = parsed_jd.get("responsibilities") or []
    if responsibilities:
        resp_lines = []
        for resp in responsibilities:
            if isinstance(resp, dict):
                category = resp.get("category") or ""
                tasks    = resp.get("tasks")    or []
                # Write each task as a full sentence with category context
                for task in tasks:
                    resp_lines.append(
                        f"Under {category}: {task}"
                    )
            else:
                resp_lines.append(resp)

        chunks.append({
            "type"    : "responsibilities",
            "content" : (
                f"The {job_title} at {company} will be responsible for "
                f"the following:\n" + "\n".join(resp_lines)
            )
        })

    # ── Chunk 4 — Qualifications ──────────────────────────────
    # Mirrors resume "qualifications" chunk (certifications mapped here)
    # Sentence format with clear must-have vs preferred distinction
    prof_qual  = parsed_jd.get("professional_qualifications") or {}
    must_quals = prof_qual.get("must_have")  or []
    pref_quals = prof_qual.get("preferred")  or []

    if must_quals or pref_quals:
        content = ""
        if must_quals:
            content += (
                f"Candidates applying for {job_title} must have: "
                f"{' '.join(must_quals)}. "
            )
        if pref_quals:
            content += (
                f"The following qualifications are preferred but not mandatory: "
                f"{' '.join(pref_quals)}."
            )
        chunks.append({
            "type"    : "qualifications",
            "content" : content.strip()
        })

    # ── Chunk 5 — Experience ──────────────────────────────────
    # NEW chunk — was missing before
    # Mirrors resume "experience" chunk → enables direct comparison
    # Pulls experience signal out of summary into its own chunk
    # so it gets the full 0.25 weight in scoring
    if exp:
        notice   = parsed_jd.get("notice_period") or ""
        content  = (
            f"The {job_title} role at {company} requires {exp} of "
            f"relevant professional experience in data science and "
            f"engineering. "
        )
        if notice:
            content += f"Preferred notice period: {notice}."

        chunks.append({
            "type"    : "experience",
            "content" : content.strip()
        })

    # ── Chunk 6 — Education ───────────────────────────────────
    # Only added if explicitly present in JD JSON
    education = parsed_jd.get("education_qualification")
    if education:
        chunks.append({
            "type"    : "education",
            "content" : (
                f"The minimum education qualification required for "
                f"{job_title} at {company} is: {education}."
            )
        })

    return chunks

In [5]:
# Load each JD individually since you don't have all_jd.json
import json

all_jd_chunks = {}

with open(r"..\output_json_jd\all_jd.json", "r") as f:
    jd_data = json.load(f)
    
for filename, jd_data in jd_data.items():
    chunks = chunk_jd(jd_data)
    all_jd_chunks[filename] = chunks
    print(f"✅ Chunked: {filename} → {len(chunks)} chunks")

✅ Chunked: DS_JD.txt → 5 chunks
✅ Chunked: PM_JD.txt → 6 chunks
✅ Chunked: SWE_JD.txt → 6 chunks


In [6]:
all_jd_chunks

{'DS_JD.txt': [{'type': 'summary',
   'content': 'We are hiring a Data Scientist at Version 1 located in Bengalore. We are seeking a Data Scientist to collaborate with a broader team comprising Data Scientists, Data Engineers, Data Analysts, Pricing Managers, and business stakeholders. Together, they will deliver a pricing optimisation process in AWS. The Data Scientist will concentrate on developing and implementing a financial model based on UK mortgage time series data. This is a Full-Time role requiring 3-5+ years of experience.'},
  {'type': 'skills',
   'content': 'The ideal candidate for Data Scientist must be proficient in the following technologies and tools: Python, PySpark, SQL, AWS services, financial principles, end-to-end data science projects. Knowledge of the following is a strong advantage: Hadoop, Spark, Tableau, Power BI, MLOps and model monitoring.'},
  {'type': 'responsibilities',
   'content': 'The Data Scientist at Version 1 will be responsible for the following:

#### Embeddings

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

# Load the model 
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    model_kwargs={"device": "cpu"},      # use "cuda" if you have GPU
    encode_kwargs={"normalize_embeddings": True}
    # normalize_embeddings=True is recommended for BGE models
    # it improves cosine similarity accuracy
)

def generate_embeddings(all_chunks):
    """
    Takes chunks dict and returns chunks with embeddings added.
    Works for both resume chunks and JD chunks.
    """
    all_embedded_chunks = {}

    for filename, chunks in all_chunks.items():
        embedded = []
        for chunk in chunks:
            content   = chunk["content"]
            embedding = embedding_model.embed_query(content)
            # embed_query() returns a plain list — no .tolist() needed

            embedded.append({
                "type"      : chunk["type"],
                "content"   : content,
                "embedding" : embedding
            })

        all_embedded_chunks[filename] = embedded
        print(f"✅ Embedded: {filename} → {len(embedded)} chunks")

    return all_embedded_chunks



Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [19]:
def embed_new_resume_chunks(
    new_chunks,
    all_embeddings_path=r"..\embeddings\all_resume_embeddings.json"
):
    all_resume_embeddings = load_json_file(all_embeddings_path, {})
    new_resume_embeddings = {}

    if not new_chunks:
        print("No new resume chunks to embed.")
        return all_resume_embeddings, new_resume_embeddings

    already_embedded = 0
    newly_embedded = 0
    failed = 0

    for filename, chunks in new_chunks.items():
        if filename in all_resume_embeddings:
            already_embedded += 1
            print(f"Skipping already embedded resume: {filename}")
            continue

        try:
            embedded = generate_embeddings({filename: chunks})[filename]

            all_resume_embeddings[filename] = embedded
            new_resume_embeddings[filename] = embedded
            newly_embedded += 1

        except Exception as e:
            failed += 1
            print(f"Failed to embed {filename}: {e}")

    save_json_file(all_embeddings_path, all_resume_embeddings)

    print("\nEmbedding Summary")
    print(f"Already embedded: {already_embedded}")
    print(f"Newly embedded: {newly_embedded}")
    print(f"Failed: {failed}")
    print(f"Total resumes in all_resume_embeddings.json: {len(all_resume_embeddings)}")

    return all_resume_embeddings, new_resume_embeddings


In [ ]:
resume_embeddings, new_resume_embeddings = embed_new_resume_chunks(new_resume_chunks)

In [24]:
with open(r"..\chunks\all_resume_chunks.json" , 'r') as f:
    data = json.load(f)

resume_embeddings = generate_embeddings(data)

✅ Embedded: Abhishek_Shaurya_Resume.pdf → 11 chunks
✅ Embedded: Jay Kumar Behera_CV_DS.pdf → 10 chunks
✅ Embedded: Komal_Kamble_1 (1).pdf → 7 chunks
✅ Embedded: MeetLad_Resume.pdf → 7 chunks
✅ Embedded: Prity-Kumari-Resume-DevOps-2025.pdf → 8 chunks
✅ Embedded: SOUMYADEEP_SEN_CV.pdf → 8 chunks
✅ Embedded: MananDaxini-Updated-2026-t (1) (2).pdf → 9 chunks


In [25]:
with open(r"..\embeddings\all_resume_embeddings.json", "w") as f:
    json.dump(resume_embeddings, f)
print("Resume embeddings saved.")

Resume embeddings saved.


In [ ]:
print(f"Total resume embeddings available: {len(resume_embeddings)}")
print(f"New resume embeddings created in this run: {len(new_resume_embeddings)}")


Total resume embeddings available: 7


In [ ]:
# Generate embeddings for JDs 
jd_embeddings = generate_embeddings(all_jd_chunks)

with open(r"..\embeddings\all_jd_embeddings.json", "w") as f:
    json.dump(jd_embeddings, f)
print("✅ JD embeddings saved.")

✅ JD embeddings saved.


In [27]:
# Lenght of resume_embeddings
len(resume_embeddings["Abhishek_Shaurya_Resume.pdf"][0]["embedding"])

1024

In [18]:
# len of jd embedding_model
len(jd_embeddings["PM_JD.txt"][0]["embedding"])

1024

Below code shows if there are any embedding mismatch

In [9]:
import json
from collections import Counter

EMBEDDINGS_PATH = r"C:\Resume_Screening_Project\embeddings\all_resume_embeddings.json"

with open(EMBEDDINGS_PATH, "r", encoding="utf-8") as f:
    resume_embeddings = json.load(f)

dims = []

for filename, chunks in resume_embeddings.items():
    for chunk in chunks:
        embedding = chunk.get("embedding", [])
        dims.append((filename, chunk.get("type"), len(embedding)))

dimension_counts = Counter(dim for _, _, dim in dims)

print("Embedding dimension counts:")
for dim, count in dimension_counts.items():
    print(f"{dim}: {count} chunks")

print("\nFiles with non-768 embeddings:")
for filename, chunk_type, dim in dims:
    if dim != 768:
        print(f"{filename} | {chunk_type} | {dim}")


Embedding dimension counts:
1024: 60 chunks

Files with non-768 embeddings:
Abhishek_Shaurya_Resume.pdf | summary | 1024
Abhishek_Shaurya_Resume.pdf | skills | 1024
Abhishek_Shaurya_Resume.pdf | experience | 1024
Abhishek_Shaurya_Resume.pdf | responsibilities | 1024
Abhishek_Shaurya_Resume.pdf | project | 1024
Abhishek_Shaurya_Resume.pdf | project | 1024
Abhishek_Shaurya_Resume.pdf | project | 1024
Abhishek_Shaurya_Resume.pdf | project | 1024
Abhishek_Shaurya_Resume.pdf | education | 1024
Abhishek_Shaurya_Resume.pdf | qualifications | 1024
Abhishek_Shaurya_Resume.pdf | contacts | 1024
Jay Kumar Behera_CV_DS.pdf | skills | 1024
Jay Kumar Behera_CV_DS.pdf | experience | 1024
Jay Kumar Behera_CV_DS.pdf | responsibilities | 1024
Jay Kumar Behera_CV_DS.pdf | project | 1024
Jay Kumar Behera_CV_DS.pdf | project | 1024
Jay Kumar Behera_CV_DS.pdf | project | 1024
Jay Kumar Behera_CV_DS.pdf | project | 1024
Jay Kumar Behera_CV_DS.pdf | education | 1024
Jay Kumar Behera_CV_DS.pdf | qualifications